A script implementing BICePs to reweight populations for a simple three state
toy model system.  Here, our prior comes from random generation of the Boltzmann
distribution and reweighting is performed using two experimental observables both set to 0.0 A.U.

For more details about this toy model systema and visual aids, please refer to
this notebook: `examples/enforcing_uniform_reference.ipynb`

In [1]:
import sys, os
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
import pandas as pd
from sklearn import metrics
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import biceps
from biceps.PosteriorSampler import u_kln_and_states_kn
from pymbar import MBAR

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



In [2]:
class Data:
    def __init__(self, array_list):
        self.array_list = array_list

    def save(self, filename):
        with open(filename, 'wb') as f:
            pickle.dump(self.array_list, f)

    @classmethod
    def load(cls, filename):
        with open(filename, 'rb') as f:
            array_list = pickle.load(f)
        return cls(array_list)

In [3]:
def write_noe_files(weights, x, exp, dir):
    for i in range(len(weights)):
        model = pd.read_pickle("template.noe")
        _model = pd.DataFrame()

        for j in range(len(exp)):
            row = model.iloc[0].copy()
            row["restraint_index"] = int(exp[j][0])
            row["atom_index1"] = int(exp[j][1])
            row["atom_index2"] = int(exp[j][2])
            row["exp"] = float(exp[j][3])
            row["model"] = float(x[i][j])  # x[i] must be flat and match len(exp)
            _model = pd.concat([_model, row.to_frame().T], ignore_index=True)

        _model.to_pickle(f"{dir}/{i}.noe")


###### Parameters #######

In [4]:
nStates,Nd = 90,1 # 76 distance with 1 NOE observables
n_xis,n_lambdas,nreplicas,nsteps,change_Nr_every,swap_every=1,2,1,1000000,0,0
multiprocess=4
σ_prior=0.161 # 0.08, 0.16
stat_model,data_uncertainty="Students","single"
data_likelihood = "gaussian" #"log normal" # "gaussian"

write_every = 10
attempt_move_state_every = 1
attempt_move_sigma_every = 1

Make output directories

In [5]:
state_dir = f"{nStates}_state"
biceps.toolbox.mkdir(state_dir)

datapoints_dir = f"{state_dir}/{nStates}_state_{Nd}_datapoints"
biceps.toolbox.mkdir(datapoints_dir)

dir = f"{datapoints_dir}/Prior_error_{σ_prior}"
biceps.toolbox.mkdir(dir)

## Loaded the population

In [6]:
clustering = pd.read_csv(f"../clustering/cluster_percentages.csv")

populations = np.array(clustering["Population"])
populations.shape

(181,)

In [7]:
energies = -np.log(populations)
energies.shape

(181,)

## Load the Prior Model (From MD) Calculated NOE distances 

In [8]:
md_distances = pd.read_csv(f"../clustering/md_distances.csv")

forward_model_data = np.array(md_distances)
forward_model_data.shape

(181, 90)

## Load the Refer NMR (Experimental Measurement)

In [9]:
restraints_table = {
    'weak': 5,
    'medium': 3.5,
    'strong': 2.5
}

dist_res_file = '../../../../utils/nspe_7_2_restraints.csv'
df_dist_res = pd.read_csv(dist_res_file)

for i, row in df_dist_res.iterrows():
    df_dist_res.at[i, df_dist_res.columns[2]] = restraints_table[row[2]]  # Correct mapping

# Make a look up table for intensity 
df_dist_res
dist_res = df_dist_res.values.tolist()
dist_res[:5]

[[160, 165, 5], [160, 166, 5], [161, 165, 5], [161, 166, 5], [137, 165, 5]]

In [10]:
np.shape(dist_res)

(90, 3)

In [11]:
### Extract the restraints_index from the labels 

human_readable_labels = "../clustering/human_readable_labels.csv"
df_labels = pd.read_csv(human_readable_labels)
df_labels[:5]
restraint_index = pd.factorize(df_labels['0'])[0] + 1

In [12]:
## Create experiment input with a list of [restraint_index, atom_index1, atom_index2, exp]

exp = [[r] + d for r, d in zip(restraint_index, dist_res)]
exp[:5]

type(exp)

list

## Write the NOE files 

In [13]:
data_dir = f"{dir}/NOE"
biceps.toolbox.mkdir(data_dir)

write_noe_files(weights=energies, x=forward_model_data, exp=exp, dir=data_dir)

In [14]:
df = pd.read_pickle(f"{data_dir}/1.noe")

df[:7]

,restraint_index,atom_index1,res1,atom_name1,atom_index2,res2,atom_name2,exp,model,comments
0,1,160,UNK1,H1,165,UNK1,H20,5.0,3.796106,NaN
1,1,160,UNK1,H1,166,UNK1,H20,5.0,3.800327,NaN
2,1,161,UNK1,H1,165,UNK1,H20,5.0,3.609056,NaN
3,1,161,UNK1,H1,166,UNK1,H20,5.0,3.595518,NaN
4,2,137,UNK1,H1,165,UNK1,H20,5.0,4.506432,NaN
5,2,137,UNK1,H1,166,UNK1,H20,5.0,4.453423,NaN
6,2,138,UNK1,H1,165,UNK1,H20,5.0,3.018879,NaN


## Load the input data

In [15]:
input_data = biceps.toolbox.sort_data(data_dir)
print(f"Input data: {biceps.toolbox.list_extensions(input_data)}")
forward_model_data = np.array([pd.read_pickle(i)["model"].to_numpy() for i in biceps.toolbox.get_files(f"{data_dir}/*.noe")])
experiment = np.array([pd.read_pickle(i)["exp"].to_numpy() for i in biceps.toolbox.get_files(f"{data_dir}/0.noe")])[0]


Input data: ['.noe']


In [16]:
outdir = f'{dir}/{stat_model}_{data_uncertainty}_sigma/{nsteps}_steps_{nreplicas}_replicas_{n_lambdas}_lam__swap_every_{swap_every}'
biceps.toolbox.mkdir(outdir)
print(f"nSteps of sampling: {nsteps}\nnReplicas: {nreplicas}")
lambda_values = np.linspace(0.0, 1.0, n_lambdas)

nSteps of sampling: 1000000
nReplicas: 1


In [17]:
sigMin,sigMax,dsig = 0.001,200,1.02
arr = np.exp(np.arange(np.log(sigMin), np.log(sigMax), np.log(dsig)))
l = len(arr)
sigma_index = round(l*0.73)

In [18]:
beta,beta_index=(1., 2.0, 1),0
_arr = np.linspace(*beta)
_l = len(_arr)
print("Alpha starts here: ",_arr[beta_index])
phi,phi_index=(1., 2.0, 1),0
gamma,gamma_index=(1.0, 2.0, np.e),0

Alpha starts here:  1.0


In [19]:
options = [dict(ref="uniform", stat_model=stat_model,
            sigma=(sigMin, sigMax, dsig), sigma_index=sigma_index, gamma=gamma,
            beta=beta, beta_index=beta_index, phi=phi, phi_index=phi_index,
            data_uncertainty=data_uncertainty, data_likelihood=data_likelihood,
            )]
print(pd.DataFrame(options))


       ref stat_model               sigma  sigma_index  \
0  uniform   Students  (0.001, 200, 1.02)          450   

                           gamma           beta  beta_index            phi  \
0  (1.0, 2.0, 2.718281828459045)  (1.0, 2.0, 1)           0  (1.0, 2.0, 1)   

   phi_index data_uncertainty data_likelihood  
0          0           single        gaussian  


In [20]:
ensemble = biceps.ExpandedEnsemble(lambda_values=lambda_values, energies=energies)
ensemble.initialize_restraints(input_data, options, verbose=1)
print("ensemble.expanded_values = ",ensemble.expanded_values)

Time to initalize restraints: 0.36s
ensemble.expanded_values =  [(0.0, 1.0), (1.0, 1.0)]


In [21]:
sampler = biceps.PosteriorSampler(ensemble, nreplicas, change_Nr_every, write_every=write_every)
sampler.sample(nsteps, attempt_lambda_swap_every=swap_every, swap_sigmas=1,
        attempt_move_state_every=attempt_move_state_every,
        attempt_move_sigma_every=attempt_move_sigma_every,
        verbose=0, progress=1, multiprocess=True, capture_stdout=0)

 ██████████████████████████████▏ 100.0% [1000000/1000000 | 55.5 kHz | 1 | 0s | 18s] MCMC 


In [22]:
expanded_values = sampler.expanded_values
A = biceps.Analysis(sampler, outdir=outdir, nstates=len(energies), MBAR=True, multiprocess=False, capture_stdout=0)
A.plot(plottype="step", figsize=(12,14), figname=f"BICePs.pdf", pad=0.35, plot_all_distributions=1)
plt.show()
A.plot_energy_trace()
plt.show()
BS, pops = A.f_df, A.P_dP[:,len(expanded_values[:])-1]
BS /= sampler.nreplicas
K = len(expanded_values[:])-1
pops_std = A.P_dP[:,2*K]
print(f"Predicted populatins: {pops}")

These states have not been sampled:
 [  9  68  70  84 103 114 149 159 180]
These states have not been sampled:
 [  9  10  11  14  15  16  19  22  24  25  27  30  32  36  38  39  40  41
  43  44  45  46  54  58  61  62  66  67  68  70  72  74  79  80  81  83
  84  85  86  87  88  89  93  94  97  98  99 100 101 102 103 105 109 110
 111 112 113 114 115 120 121 122 124 125 126 127 128 129 130 131 135 136
 140 141 142 143 144 147 148 149 150 151 152 154 155 157 158 159 160 164
 165 166 168 169 170 171 173 175 176 177 178 179 180]
                               ▏  0.0% [   0/100000 | 0.0 Hz | 1 | infs | 0s] u_kln 


******* JAX 64-bit mode is now on! *******
*     JAX is now set to 64-bit mode!     *
*   This MAY cause problems with other   *
*      uses of JAX in the same code.     *
******************************************



 ██████████████████████████████▏ 100.0% [100000/100000 | 247.7 kHz | 1 | 0s | 0s] u_kln 
Time for MBAR: 6.477 s
Writing 90_state/90_state_1_datapoints/Prior_error_0.161/Students_single_sigma/1000000_steps_1_replicas_2_lam__swap_every_0/BS.dat...
Writing 90_state/90_state_1_datapoints/Prior_error_0.161/Students_single_sigma/1000000_steps_1_replicas_2_lam__swap_every_0/populations.dat...
Top 18 states: [117, 35, 134, 73, 77, 78, 12, 8, 75, 104, 28, 48, 2, 71, 0, 7, 49, 1]
Top 18 populations: [0.00122219 0.00130029 0.00156853 0.00190491 0.00242669 0.00335102
 0.00355969 0.00442308 0.00456375 0.00458596 0.00757986 0.00996561
 0.01609135 0.01760799 0.01955714 0.0309827  0.04718035 0.81199177]
nplots =  1


/var/folders/d8/y2dvs1ln1gjcwccrkvtffr240000gn/T/ipykernel_85367/3964779525.py:4: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Predicted populatins: [1.95571360e-02 8.11991767e-01 1.60913461e-02 1.39763970e-04
 6.71420204e-05 3.20637343e-05 2.05414114e-04 3.09826987e-02
 4.42308012e-03 0.00000000e+00 3.74897975e-09 3.75126788e-06
 3.55969370e-03 5.19502910e-05 2.91805415e-06 1.24965992e-09
 2.51136877e-06 4.88811119e-05 2.90637021e-04 2.24910679e-08
 1.97169341e-05 1.10992671e-05 3.24596398e-07 4.13530288e-04
 1.26965593e-06 3.99741311e-08 2.74447789e-05 7.13473755e-07
 7.57986218e-03 9.13449869e-04 1.79203174e-07 2.95367547e-04
 2.81323855e-06 7.20605120e-05 1.94263808e-04 1.30028751e-03
 2.55732204e-06 1.30575483e-05 2.44838576e-06 3.43079643e-06
 7.02387627e-07 2.97723268e-07 4.89537063e-04 8.62265343e-08
 3.74897975e-08 2.49900754e-08 3.74493984e-06 2.20575921e-04
 9.96561283e-03 4.71803505e-02 2.40208732e-04 2.67574541e-05
 1.09732196e-03 1.13361242e-05 3.48789469e-06 7.02316245e-04
 6.29110401e-05 1.43320839e-04 3.90616420e-07 1.52907151e-05
 1.07735089e-05 3.84606878e-07 4.71193785e-07 1.27152317e-05
 4

In [23]:
pops.shape

# Convert to DataFrame
df_predicted_population = pd.DataFrame(pops)

# Save to CSV
df_predicted_population.to_csv("../clustering/predicted_population_biceps.csv", index=False)

In [24]:
most_populated_index = np.argmax(pops)
most_populated_value = pops[most_populated_index]

print(f"Most populated state: {most_populated_index} with value {most_populated_value}")


Most populated state: 1 with value 0.8119917668170461


In [25]:
top5_indices = np.argsort(pops)[-5:][::-1]  # Sort, take last 5, reverse for descending order
top5_values = pops[top5_indices]

for i, (idx, val) in enumerate(zip(top5_indices, top5_values), 1):
    print(f"Top {i}: index = {idx}, population = {val:.4f}")

top5_indices

Top 1: index = 1, population = 0.8120
Top 2: index = 49, population = 0.0472
Top 3: index = 7, population = 0.0310
Top 4: index = 0, population = 0.0196
Top 5: index = 71, population = 0.0176


array([ 1, 49,  7,  0, 71])